# Day 3 — ILT 2: CDC Concepts — WAL and Logical Replication from Supabase
**Time:** 12:00 PM – 1:00 PM  
**What comes next:** Hands-on — Set up CDC on Supabase, validate a captured change (2:00–4:00 PM)  

---
### What we cover today
1. What is CDC and why we need it (not just incremental load)
2. How PostgreSQL WAL works — the source of all CDC
3. Logical replication in Supabase — enabling it
4. Connecting Databricks to Supabase via JDBC
5. Reading tables from Supabase directly into Spark

> **Instructor note:** 60 minutes. ~20 min on CDC theory (draw on board: WAL → Debezium → Kafka OR direct), ~20 min JDBC connection live demo, ~15 min WAL/logical replication setup in Supabase UI, ~5 min wrap-up.

## Section 1 — Why CDC? Isn't Incremental Load Enough?

Recall from Day 1: **Incremental Load** reads only new rows using a watermark (date filter).  
It works for **inserts only**. But what about **updates and deletes**?

### The Problem with Incremental Load for Updates

```
customers table in Supabase:

Day 1:  CustomerID=C001, email=raj@old.com,   city=Mumbai    ← loaded to Bronze
Day 5:  CustomerID=C001, email=raj@new.com,   city=Delhi     ← customer UPDATED

Incremental Load with date filter:
  filter: WHERE created_at > last_run
  → C001 was created before last_run, so it is SKIPPED
  → Bronze still has raj@old.com and Mumbai — WRONG!
```

**CDC captures the UPDATE** — it logs that C001 changed from Mumbai to Delhi.  
That change gets propagated to Bronze → Silver → Gold automatically.

### The 3 change types CDC captures

| Operation | Example | Result in Lakehouse |
|-----------|---------|--------------------|
| **INSERT** | New customer signs up | New row added |
| **UPDATE** | Customer changes city | Existing row updated (or history tracked) |
| **DELETE** | Customer deletes account | Row removed (or marked deleted) |

## Section 2 — PostgreSQL WAL (Write-Ahead Log)

### What is WAL?

**WAL = Write-Ahead Log** — a file where PostgreSQL records every change BEFORE it commits it to the actual table.

Think of it like a bank's transaction journal:
```
Bank transaction journal:
  10:00 — Raj transferred ₹1000 to Priya
  10:01 — Account balance updated: Raj -₹1000, Priya +₹1000
  10:02 — New customer opened account

PostgreSQL WAL:
  10:00 — INSERT into customers: {id=C001, name=Raj, city=Mumbai}
  10:01 — UPDATE customers: C001 city changed Mumbai → Delhi
  10:02 — DELETE from customers: C999 removed
```

**WAL's original purpose:** crash recovery — if Postgres crashes, it replays the WAL to restore data.  
**CDC uses WAL:** we read the WAL stream instead of querying the table, so we capture EVERY change.

### Logical Replication — making WAL readable

By default, WAL is in a binary format only Postgres understands.  
**Logical replication** decodes WAL into human-readable change events (INSERT/UPDATE/DELETE).

```
WAL (binary):      0x4F02A1B4...
After decoding:    {"op": "UPDATE", "table": "customers", "id": "C001", "city": {"old": "Mumbai", "new": "Delhi"}}
```

### How to enable Logical Replication in Supabase

Supabase makes this easy through the Dashboard:

```
Supabase Dashboard
  → Database
  → Replication
  → Enable Row Level Changes for the tables you want to track
  (This sets wal_level = logical in PostgreSQL config)
```

> **We will do this in the hands-on (2:00 PM).**  
> For now — understand that enabling this is a one-time setup in Supabase UI.

## Section 3 — CDC Architecture for GlobalMart

### Full CDC Pipeline

```
Supabase PostgreSQL
       |
   WAL stream (logical replication)
       |
       v
  [Option A: Debezium → Kafka → Databricks]   ← Production grade, complex
  [Option B: Direct JDBC polling]              ← Simpler, works for bootcamp
       |
       v
  Bronze (Delta)  — raw CDC events or latest snapshot
       |
       v
  Silver — MERGE new/changed rows into clean table
       |
       v
  Gold  — aggregations always reflect latest customer data
```

### For this bootcamp — JDBC polling (simplified CDC)

Full CDC with Debezium + Kafka is complex to set up. Instead we use:  
**JDBC + `updated_at` watermark** — read rows from Supabase WHERE `updated_at > last_run`.  

This captures inserts AND updates (as long as tables have an `updated_at` column).  
Deletes are handled separately (soft deletes or a separate scan).

| Approach | Captures Inserts | Captures Updates | Captures Deletes | Complexity |
|----------|-----------------|-----------------|-----------------|------------|
| Full Load | Yes (all) | Yes (all) | Yes (by full replace) | Low |
| Incremental (created_at) | Yes | No | No | Low |
| **JDBC + updated_at** | Yes | Yes | Partial (soft delete) | Low-Medium |
| Debezium + Kafka | Yes | Yes | Yes | High |

## Setup — ADLS + Supabase Credentials

In [ ]:
# ─── ADLS Setup ───────────────────────────────────────────────────────────────
storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← Replace

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

# ─── Supabase (PostgreSQL) Credentials ───────────────────────────────────────
# WARNING: Do NOT commit this notebook to GitHub with the real password filled in
# In production, use Databricks Secrets: dbutils.secrets.get(scope, key)

SUPABASE_HOST     = "aws-0-ap-south-1.pooler.supabase.com"
SUPABASE_PORT     = "5432"
SUPABASE_DB       = "postgres"
SUPABASE_USER     = "postgres.isqcnhvlfnjszllicxqi"
SUPABASE_PASSWORD = "YOUR_SUPABASE_PASSWORD"  # ← paste from manager's credentials

# JDBC URL — standard PostgreSQL format
jdbc_url = f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

# Connection properties — always include ssl=true for Supabase
connection_properties = {
    "user"     : SUPABASE_USER,
    "password" : SUPABASE_PASSWORD,
    "driver"   : "org.postgresql.Driver",
    "ssl"      : "true",
    "sslmode"  : "require"
}

print("JDBC URL:", jdbc_url)
print("Credentials set — ready to connect to Supabase")

## Section 4 — Reading from Supabase via JDBC

In [ ]:
# ─── Read customers table directly from Supabase PostgreSQL ──────────────────
# This is a DIRECT database read — no CSV file needed!
# Spark connects to Supabase, runs a SELECT, and returns a DataFrame

# NOTE: The PostgreSQL JDBC driver must be installed on your Databricks cluster
# Go to: Cluster → Libraries → Install New → Maven → search 'postgresql'
# Package: org.postgresql:postgresql:42.6.0

try:
    customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = "customers",   # table name in Supabase
            properties = connection_properties
        )

    print(f"Rows read from Supabase customers: {customers_df.count():,}")
    print("\nSchema from Supabase:")
    customers_df.printSchema()
    customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting steps:")
    print("  1. Make sure the PostgreSQL JDBC driver is installed on the cluster")
    print("  2. Check that the password is correct")
    print("  3. Try: Cluster → Libraries → Install: org.postgresql:postgresql:42.6.0")

In [ ]:
# ─── Read with a SQL query instead of full table ──────────────────────────────
# You can push SQL down to Postgres — only read what you need
# Wrap the query in parentheses and alias it as 'query'

# This reads only customers updated in the last 30 days
# This is the CDC/incremental pattern — read only changed rows
query = """
    (SELECT *
     FROM customers
     WHERE updated_at > NOW() - INTERVAL '30 days'
     ORDER BY updated_at DESC) AS recent_customers
"""

try:
    recent_customers_df = spark.read \
        .jdbc(
            url        = jdbc_url,
            table      = query,
            properties = connection_properties
        )

    print(f"Customers updated in last 30 days: {recent_customers_df.count():,}")
    recent_customers_df.show(5, truncate=True)

except Exception as e:
    print(f"Query failed: {e}")

In [ ]:
# ─── Simulate CDC: read all 10 tables from Supabase and land in Bronze ────────
# In production, this is what the ingestion pipeline does:
#   1. Connect to Supabase via JDBC
#   2. Read each table (full load or watermark filter)
#   3. Save to Bronze as Delta

tables_to_ingest = [
    "customers",
    "orders",
    "orders_items",
    "products",
    "payments",
    "addresses",
    "returns",
    "suppliers",
    "payment_methods",
    "shipping_tier"
]

print(f"{'Table':<25} {'Rows':>10}  Status")
print("-" * 50)

for table_name in tables_to_ingest:
    try:
        df = spark.read.jdbc(
            url        = jdbc_url,
            table      = table_name,
            properties = connection_properties
        )
        row_count = df.count()

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .save(f"{bronze_path}/{table_name}")

        print(f"{table_name:<25} {row_count:>10,}  Saved to Bronze from Supabase")

    except Exception as e:
        print(f"{table_name:<25} {'':>10}  ERROR: {str(e)[:60]}")

print("-" * 50)
print("When this works, you no longer need CSV files — data comes direct from Supabase!")

## Section 5 — Enabling Logical Replication in Supabase (hands-on prep)

For the hands-on session at 2:00 PM, we need to enable logical replication in Supabase.

### Steps in Supabase Dashboard

```
1. Log in to Supabase → your project

2. Go to: Database → Replication

3. Under "Source" — you will see your tables listed

4. Toggle ON the tables you want CDC for:
   ✅ customers      (address + email changes)
   ✅ orders         (status changes: pending → shipped → delivered)
   ✅ addresses      (customers move)

5. This sets: wal_level = logical in PostgreSQL config
   Supabase handles the rest automatically
```

### What happens after enabling

```
Someone updates a customer's city in the app
    ↓
PostgreSQL writes the change to WAL
    ↓
Logical replication decodes it:
    {"op": "UPDATE", "table": "customers", "id": "C001",
     "city": {"old": "Mumbai", "new": "Delhi"}}
    ↓
We read this change event into Databricks
    ↓
MERGE into Silver → Gold reflects the new city
```

> **In the hands-on:** You will make a change in Supabase (update a customer row),  
> then read it back in Databricks and show the change was captured.

## Recap

| Topic | Key Takeaway |
|-------|--------------|
| Why CDC | Incremental load misses UPDATEs and DELETEs — CDC captures all 3 |
| WAL | PostgreSQL's change log — the source of all CDC |
| Logical replication | Decodes WAL into readable INSERT/UPDATE/DELETE events |
| JDBC | Direct database connection — reads tables from Supabase into Spark |
| JDBC + `updated_at` | Simple CDC pattern — reads rows changed since last run |
| Supabase setup | Enable logical replication in Dashboard → Database → Replication |

---

## Hands-on Next (2:00 PM — 2 hours)

1. Enable logical replication for `customers` and `orders` in Supabase Dashboard
2. Connect Databricks to Supabase via JDBC — read all 10 tables
3. Update a row in Supabase (e.g., change a customer's city)
4. Read the updated row back in Databricks → confirm the change was captured
5. Save the CDC snapshot to Bronze Delta

---

## Next ILT — Day 3 ILT 3 (4:00 PM)
**Change Data Feed (CDF) in Databricks — tracking changes inside Delta tables**